<a href="https://colab.research.google.com/github/felikstriagti42-boop/SISTEM-SIJADWAL/blob/main/Salinan_dari_UTS_AI_FELIKS_TRI_AGTI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# INSTAL LIBRARY TERLEBIH DAHULU

# ===== SETUP: library dan embedding =====
SHOT = {'COLAB': (1, 'Halaman Colab (judul notebook, menu Runtime)'), 'A4': (2, 'A.4 Praktik privasi dan minimisasi data'), 'A5': (3, 'A.5 Audit bias (FPR per kelompok)'), 'SETUP12': (4, 'SETUP: library dan embedding'), 'SETUP3': (5, 'SETUP 3: PostgreSQL dan pgvector'), 'B1': (6, 'B.1 Data mentah OLTP'), 'B2': (7, 'B.2 Preparasi data'), 'C1': (8, 'C.1 Skema products dan indeks'), 'C2': (9, 'C.2 Hybrid Search dengan RRF'), 'C3': (10, 'C.3 Evaluasi keyword, vector, hybrid'), 'C4': (11, 'C.4 Verifikasi RRF, EXPLAIN, ef_search'), 'C5': (12, 'C.5 Pembobotan alpha vs RRF'), 'D2': (13, 'D.2 Chunking dan ChromaDB'), 'D3': (14, 'D.3 Evaluasi retrieval'), 'D4': (15, 'D.4 Jawaban RAG'), 'D6': (16, 'D.6 Parent-Document Retrieval'), 'E1': (17, 'E.1 Perhitungan fitur'), 'E2': (18, 'E.2 Feast: apply dan materialize'), 'E3': (19, 'E.3 Point-in-Time Join'), 'E4': (20, 'E.4 Evaluasi model'), 'E5': (21, 'E.5 Inference dan latensi'), 'E7': (22, 'E.7 Data drift (Evidently)'), 'E8': (23, 'E.8 Latihan kasus Materi 2'), 'F': (24, 'F. Ringkasan hasil')}
def shot(k): print(f"[Gambar {SHOT[k][0]}] {SHOT[k][1]}\n")
shot("SETUP12")

# ===== SETUP 1: library =====
import sys, os, subprocess, time, importlib, importlib.util, warnings, re, json, random, textwrap
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.returncode

NEED = {"chromadb":"chromadb","rank_bm25":"rank_bm25","langchain_text_splitters":"langchain-text-splitters",
        "feast":"feast","evidently":"evidently","psycopg2":"psycopg2-binary","pyarrow":"pyarrow","sklearn":"scikit-learn"}
missing = [p for m,p in NEED.items() if importlib.util.find_spec(m) is None]
if IN_COLAB and importlib.util.find_spec("sentence_transformers") is None:
    missing.append("sentence-transformers")
if missing:
    print("Menginstal:", missing)
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], capture_output=True, text=True)
    if r.returncode: print("PERINGATAN pip:", r.stderr[-1200:])
    print("Selesai. Jika ada error import setelah ini: Runtime > Restart session, lalu Run all.")

import numpy as np, pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
random.seed(42); np.random.seed(42)
print("Siap. Colab:", IN_COLAB)

# ===== SETUP 2: Embedding (model asli bila tersedia, jika tidak -> fallback offline) =====
DIM = 384
_STOP = set("yang dan di ke dari untuk dengan atau the a of to in is".split())
_SYN = {"notebook":"laptop","hp":"ponsel","smartphone":"ponsel","refund":"pengembalian","retur":"pengembalian",
        "kembalikan":"pengembalian","second":"bekas","seken":"bekas","rog":"gaming","murah":"hemat","ekonomis":"hemat"}
import hashlib
def hash_embed(texts):
    # Fallback offline: hashing-trick + sinonim sederhana, L2-normalized, 384 dimensi.
    out = []
    for t in texts:
        v = np.zeros(DIM, dtype=np.float32)
        for w in [x for x in re.findall(r"[a-z0-9]+", t.lower()) if x not in _STOP]:
            w = _SYN.get(w, w); h = int(hashlib.md5(w.encode()).hexdigest(), 16)
            v[h % DIM] += 1.0 if (h >> 8) % 2 else -1.0
        n = np.linalg.norm(v); out.append((v/n if n else v).tolist())
    return out

try:
    from sentence_transformers import SentenceTransformer
    _m = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")   # 384 dim, mendukung Bahasa Indonesia
    assert _m.get_sentence_embedding_dimension() == DIM
    def embed(texts): return _m.encode(list(texts), normalize_embeddings=True, batch_size=64).tolist()
    EMB_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
except Exception as e:
    embed = hash_embed
    EMB_NAME = "hash_embed (fallback offline, bukan model semantik sungguhan)"
print("Embedding:", EMB_NAME)


[Gambar 4] SETUP: library dan embedding

Siap. Colab: True


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [ ]:

shot("SETUP3")
# ===== SETUP 3: PostgreSQL + pgvector =====
import psycopg2, psycopg2.extras
PG = dict(host="localhost", port=5432, user="postgres", password="postgres")
def pg_connect(db="postgres"):
    c = psycopg2.connect(dbname=db, **PG); c.autocommit = True; return c
def pg_alive():
    try: pg_connect().close(); return True
    except Exception: return False

if not pg_alive():
    print("Menginstal PostgreSQL ...")
    sh("apt-get -qq update"); sh("DEBIAN_FRONTEND=noninteractive apt-get -qq install -y postgresql postgresql-contrib")
    sh("service postgresql start")
    for _ in range(20):                       # tunggu server siap (maks ~40 detik)
        if pg_alive(): break
        time.sleep(2)
    sh('''su postgres -c "psql -c \\"ALTER USER postgres PASSWORD 'postgres';\\""''')
assert pg_alive(), "PostgreSQL belum berjalan"

PGV = sorted(os.listdir("/usr/lib/postgresql"))[-1]
ctl = f"/usr/share/postgresql/{PGV}/extension/vector.control"
if not os.path.exists(ctl):
    sh("apt-get -qq update")
    sh(f"DEBIAN_FRONTEND=noninteractive apt-get -qq install -y postgresql-{PGV}-pgvector")
if not os.path.exists(ctl):   # fallback: build dari sumber
    print("Build pgvector dari sumber ...")
    sh(f"DEBIAN_FRONTEND=noninteractive apt-get -qq install -y postgresql-server-dev-{PGV} build-essential git")
    sh("rm -rf /tmp/pgvector && git clone -q --branch v0.8.0 https://github.com/pgvector/pgvector.git /tmp/pgvector")
    sh("cd /tmp/pgvector && make -s && make -s install")
assert os.path.exists(ctl), "pgvector gagal terpasang"

c0 = pg_connect(); cur0 = c0.cursor()
cur0.execute("SELECT 1 FROM pg_database WHERE datname='tokopintar'")
if not cur0.fetchone(): cur0.execute("CREATE DATABASE tokopintar")
c0.close()
conn = pg_connect("tokopintar"); cur = conn.cursor()
cur.execute("CREATE EXTENSION IF NOT EXISTS vector")
cur.execute("SELECT version(), (SELECT extversion FROM pg_extension WHERE extname='vector')")
v = cur.fetchone(); print(v[0].split(",")[0], "| pgvector", v[1])


# ---------- Penjelasan: BAGIAN B — Data Mentah (OLTP) & Preparasi Basis Data ----------

[Gambar 5] SETUP 3: PostgreSQL dan pgvector

PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1) on x86_64-pc-linux-gnu | pgvector 0.6.0


In [ ]:

shot("B1")
# ---------- Pengguna ----------
N_USERS = 300
tiers = np.random.choice(["Tier1","Tier2","Tier3"], N_USERS, p=[.4,.35,.25])
users = pd.DataFrame({
    "user_id": np.arange(1, N_USERS+1),
    "city_tier": tiers,
    "email": [f"Felikstriagti{i}@mail.co.id" for i in range(1, N_USERS+1)],
    "phone": [f"08{random.randint(10,99)}{random.randint(10000000,99999999)}" for _ in range(N_USERS)],
})
base_mean = {"Tier1":90000, "Tier2":70000, "Tier3":55000}

# ---------- Transaksi (60 hari sebelum 2026-10-05) ----------
END = pd.Timestamp("2026-10-05 12:00:00", tz="UTC"); START = END - pd.Timedelta(days=60)
rows, tx_id = [], 0
for u in users.itertuples():
    mu = base_mean[u.city_tier]; devs = [f"DEV_{u.user_id}_A"] + ([f"DEV_{u.user_id}_B"] if random.random() < .15 else [])
    for _ in range(random.randint(20, 55)):
        ts = START + pd.Timedelta(seconds=random.randint(0, int((END-START).total_seconds())))
        amt = int(np.random.lognormal(np.log(mu), 0.45))
        st = np.random.choice(["SUCCESS","FAILED","PENDING"], p=[.92,.05,.03])
        tx_id += 1; rows.append([f"TX_{tx_id:06d}", u.user_id, amt, ts, random.choice(devs), st, 0])
# ---------- Suntik pola fraud: lonjakan nilai + burst beberapa transaksi dalam 1 jam ----------
fraud_users = np.random.choice(users.user_id, 40, replace=False)
for uid in fraud_users:
    mu = base_mean[users.loc[users.user_id==uid, "city_tier"].iloc[0]]
    t0 = START + pd.Timedelta(seconds=random.randint(int(20*86400), int((END-START).total_seconds())-7200))
    for k in range(random.randint(2, 4)):
        tx_id += 1
        rows.append([f"TX_{tx_id:06d}", int(uid), int(mu*random.uniform(30, 100)), t0 + pd.Timedelta(minutes=12*k),
                     f"DEV_{uid}_NEW", "SUCCESS", 1])
raw = pd.DataFrame(rows, columns=["transaction_id","user_id","amount","created_at","device_id","status","is_fraud_label"])

# ---------- Kotoran data ----------
dup = raw.sample(frac=0.02, random_state=1); raw = pd.concat([raw, dup], ignore_index=True)       # duplikat
raw["amount_raw"] = raw["amount"].astype(str)
txt = raw.sample(frac=0.05, random_state=2).index
raw.loc[txt, "amount_raw"] = raw.loc[txt, "amount"].map(lambda x: "Rp " + f"{x:,}".replace(",", "."))  # "Rp 50.000"
raw.loc[raw.sample(frac=0.01, random_state=3).index, "device_id"] = None                           # device kosong
raw = raw.sort_values("created_at").reset_index(drop=True)

# ---------- Tulis ke PostgreSQL ----------
cur.execute("DROP TABLE IF EXISTS raw_transactions; DROP TABLE IF EXISTS users;")
cur.execute('''CREATE TABLE users(user_id INT PRIMARY KEY, city_tier TEXT, email TEXT, phone TEXT);''')
cur.execute('''CREATE TABLE raw_transactions(transaction_id TEXT, user_id INT, amount_raw TEXT,
               created_at TIMESTAMPTZ, device_id TEXT, status TEXT, is_fraud_label SMALLINT);''')
psycopg2.extras.execute_values(cur, "INSERT INTO users VALUES %s", list(users.itertuples(index=False, name=None)))
psycopg2.extras.execute_values(cur, "INSERT INTO raw_transactions VALUES %s",
    [(r.transaction_id, int(r.user_id), r.amount_raw, r.created_at.to_pydatetime(), r.device_id, r.status, int(r.is_fraud_label))
     for r in raw.itertuples()])
cur.execute("SELECT status, COUNT(*) FROM raw_transactions GROUP BY status ORDER BY 2 DESC")
print("Baris mentah per status:", cur.fetchall())
cur.execute("SELECT * FROM raw_transactions ORDER BY created_at LIMIT 5"); print(pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description]))


# ---------- Penjelasan: B.2 Preparasi basis data (Data Cleansing & Transformation) -

[Gambar 6] B.1 Data mentah OLTP

Baris mentah per status: [('SUCCESS', 10766), ('FAILED', 584), ('PENDING', 346)]
  transaction_id  user_id amount_raw                created_at  device_id   status  is_fraud_label
0      TX_005047      135      93408 2026-08-06 12:04:59+00:00  DEV_135_A  SUCCESS               0
1      TX_007569      199      45508 2026-08-06 12:12:59+00:00  DEV_199_A  SUCCESS               0
2      TX_002225       61  Rp 23.251 2026-08-06 12:27:21+00:00   DEV_61_A  SUCCESS               0
3      TX_006806      180      76502 2026-08-06 12:37:51+00:00  DEV_180_A  SUCCESS               0
4      TX_008467      224  Rp 97.200 2026-08-06 12:47:59+00:00  DEV_224_A   FAILED               0


In [ ]:

shot("B2")
def read_sql_df(q):
    cur.execute(q); return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])

df = read_sql_df("SELECT * FROM raw_transactions")
report = {"baris_mentah": len(df)}

# 1) Deduplikasi berdasarkan transaction_id
df = df.drop_duplicates("transaction_id", keep="first"); report["setelah_dedup"] = len(df)
# 2) Normalisasi amount: "Rp 50.000" -> 50000 (integer murni)
df["amount"] = df["amount_raw"].str.replace(r"[^0-9]", "", regex=True).astype("int64")
# 3) Status: FAILED dibuang; PENDING tidak boleh masuk agregat historis
report["failed_dibuang"] = int((df.status=="FAILED").sum()); report["pending_ditahan"] = int((df.status=="PENDING").sum())
pending = df[df.status=="PENDING"].copy()
clean = df[df.status=="SUCCESS"].copy()
# 4) Nilai kosong: device -> 'UNKNOWN'
report["device_kosong"] = int(clean.device_id.isna().sum()); clean["device_id"] = clean.device_id.fillna("UNKNOWN")
# 5) Waktu -> UTC konsisten
clean["created_at"] = pd.to_datetime(clean["created_at"], utc=True)
clean = clean.sort_values(["user_id","created_at"]).reset_index(drop=True)
report["baris_bersih_SUCCESS"] = len(clean)

# 6) Validasi kualitas otomatis (gaya Great Expectations; assert sederhana)
assert clean.transaction_id.is_unique, "ID duplikat"
assert (clean.amount > 0).all(), "amount tidak valid"
assert clean.status.eq("SUCCESS").all()
assert clean.created_at.dt.tz is not None
print(pd.Series(report).to_string())
print("\nValidasi kualitas: LULUS")


[Gambar 7] B.2 Preparasi data

baris_mentah            11696
setelah_dedup           11467
failed_dibuang            575
pending_ditahan           342
device_kosong             105
baris_bersih_SUCCESS    10550

Validasi kualitas: LULUS


In [ ]:

shot("A4")
# ===== A.4 Praktik privasi & minimisasi data (PII tidak masuk pipeline fitur / prompt LLM) =====
def mask_email(e):  a, d = e.split("@"); return a[0] + "***@" + d
def mask_phone(p):  return p[:4] + "****" + p[-4:]
view = users.head(3).copy()
view["email_masked"] = view.email.map(mask_email); view["phone_masked"] = view.phone.map(mask_phone)
print(view[["user_id","email","email_masked","phone","phone_masked"]].to_string(index=False))

# Hanya kolom non-PII yang boleh masuk dataset fitur. Pemeriksaan otomatis:
PII_COLS = {"email","phone"}
FEATURE_INPUT_COLS = ["user_id","transaction_id","amount","created_at","device_id","status"]
assert PII_COLS.isdisjoint(FEATURE_INPUT_COLS)
print("\nCek minimisasi data: kolom PII tidak ada pada input pipeline fitur -> LULUS")


# ---------- Penjelasan: BAGIAN C — TUGAS 1: Hybrid Search (BM25/Full-Text + pgvector) di PostgreSQL ----------


[Gambar 2] A.4 Praktik privasi dan minimisasi data

 user_id                     email    email_masked        phone phone_masked
       1 Felikstriagti1@mail.co.id F***@mail.co.id 089124942603 0891****2603
       2 Felikstriagti2@mail.co.id F***@mail.co.id 081346913810 0813****3810
       3 Felikstriagti3@mail.co.id F***@mail.co.id 084139958838 0841****8838

Cek minimisasi data: kolom PII tidak ada pada input pipeline fitur -> LULUS


In [ ]:

shot("C1")
# ---------- Katalog produk sintetis (dengan atribut terstruktur untuk evaluasi objektif) ----------
prods = []
lines = {"ASUS":["ROG Strix","VivoBook"], "Lenovo":["Legion","IdeaPad"], "Acer":["Aspire","Predator"], "HP":["Pavilion","Omen"], "Dell":["Inspiron","G15"]}
for b, ls in lines.items():
    for ln in ls:
        for cpu in ["i5","i7","Ryzen 5","Ryzen 7"]:
            for ram in [8,16,32]:
                for cond in ["baru","bekas"]:
                    word = random.choice(["Laptop","Notebook"])
                    cond_desc = "baru garansi resmi" if cond=="baru" else random.choice(["bekas terawat","second mulus","seken kondisi baik"])
                    extra = " Republic of Gamers" if ln=="ROG Strix" and random.random()<.7 else ""
                    price = int(6e6 + ram*2.5e5 + (3e6 if "ROG" in ln or ln in ("Legion","Predator","Omen","G15") else 0)
                                + (1e6 if "7" in cpu else 0)) * (0.65 if cond=="bekas" else 1)
                    prods.append(dict(name=f"{b} {ln} {cpu} RAM {ram}GB {cond}",
                        description=f"{word} {b} {ln}{extra} prosesor {cpu} memori {ram}GB SSD 512GB {cond_desc}",
                        category="Laptop", brand=b, line=ln, cpu=cpu, ram_gb=ram, cond=cond, gender=None, price=price))
for b, ms in {"Nike":["Pegasus","Revolution"], "Adidas":["Ultraboost","Duramo"], "Puma":["Velocity","Deviate"], "Asics":["Gel Nimbus","Gel Kayano"]}.items():
    for m in ms:
        for g in ["Pria","Wanita"]:
            prods.append(dict(name=f"Sepatu Lari {b} {m} {g}", description=f"Sepatu lari {g.lower()} {b} {m} bantalan empuk nyaman untuk maraton",
                category="Sepatu", brand=b, line=m, cpu=None, ram_gb=None, cond="baru", gender=g, price=random.randint(8,28)*100000))
for b, ms in {"Samsung":["Galaxy A55","Galaxy S24"], "Xiaomi":["Redmi Note 13","Poco X6"], "Oppo":["Reno 11","A78"]}.items():
    for m in ms:
        for ram in [8,12]:
            word = random.choice(["Smartphone","HP","Ponsel"])
            prods.append(dict(name=f"{b} {m} RAM {ram}GB", description=f"{word} {b} {m} RAM {ram}GB kamera 50MP baterai awet",
                category="Ponsel", brand=b, line=m, cpu=None, ram_gb=ram, cond="baru", gender=None, price=random.randint(20,150)*100000))
P = pd.DataFrame(prods); P["price"] = P.price.astype(int); P.insert(0, "id", range(1, len(P)+1))
print("Jumlah produk:", len(P), "|", P.category.value_counts().to_dict())

# ---------- Skema hybrid ----------
cur.execute("DROP TABLE IF EXISTS products")
cur.execute(f'''CREATE TABLE products(
  id BIGINT PRIMARY KEY, name TEXT NOT NULL, description TEXT NOT NULL,
  category TEXT, brand TEXT, line TEXT, cpu TEXT, ram_gb INT, cond TEXT, gender TEXT, price NUMERIC(14,0),
  fts_tokens TSVECTOR GENERATED ALWAYS AS (to_tsvector('simple', name || ' ' || description)) STORED,
  embedding VECTOR({DIM}))''')
t0 = time.time(); E = embed((P.name + ". " + P.description).tolist()); print(f"Embedding {len(E)} produk: {time.time()-t0:.1f}s")
def nn(x): return None if pd.isna(x) else x
psycopg2.extras.execute_values(cur,
  "INSERT INTO products(id,name,description,category,brand,line,cpu,ram_gb,cond,gender,price,embedding) VALUES %s",
  [(int(r.id), r.name, r.description, r.category, r.brand, r.line, nn(r.cpu), None if pd.isna(r.ram_gb) else int(r.ram_gb), r.cond, nn(r.gender), int(r.price), str(e))
   for r, e in zip(P.itertuples(), E)], template="(%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s::vector)")
cur.execute("CREATE INDEX idx_products_fts ON products USING GIN(fts_tokens) WITH (fastupdate = off)")
cur.execute("CREATE INDEX idx_products_hnsw ON products USING hnsw (embedding vector_cosine_ops) WITH (m = 16, ef_construction = 64)")
cur.execute("CREATE INDEX idx_products_cat ON products(category)")
cur.execute("ANALYZE products")
cur.execute("SELECT indexname FROM pg_indexes WHERE tablename='products' ORDER BY 1"); print("Indeks:", [r[0] for r in cur.fetchall()])


# ---------- Penjelasan: C.2 Kueri Hybrid Search + RRF (satu kueri SQL) ----------

[Gambar 8] C.1 Skema products dan indeks

Jumlah produk: 268 | {'Laptop': 240, 'Sepatu': 16, 'Ponsel': 12}
Embedding 268 produk: 9.8s
Indeks: ['idx_products_cat', 'idx_products_fts', 'idx_products_hnsw', 'products_pkey']


In [ ]:

shot("C2")
def tsq(q):
    toks = [t for t in re.findall(r"[a-z0-9]+", q.lower()) if t not in _STOP]
    return " | ".join(dict.fromkeys(toks))

HYBRID_SQL = '''
WITH fts AS (
  SELECT id, ROW_NUMBER() OVER (ORDER BY ts_rank(fts_tokens, q) DESC, id) AS rnk
  FROM products, to_tsquery('simple', %(tsq)s) q
  WHERE fts_tokens @@ q AND (%(cat)s::text IS NULL OR category = %(cat)s)
  ORDER BY ts_rank(fts_tokens, q) DESC, id LIMIT 20),
vec AS (
  SELECT id, ROW_NUMBER() OVER (ORDER BY embedding <=> %(vec)s::vector) AS rnk
  FROM products WHERE (%(cat)s::text IS NULL OR category = %(cat)s)
  ORDER BY embedding <=> %(vec)s::vector LIMIT 20)
SELECT p.id, p.name, p.price, fts.rnk AS rank_kw, vec.rnk AS rank_vec,
       COALESCE(1.0/(60+fts.rnk),0) + COALESCE(1.0/(60+vec.rnk),0) AS rrf
FROM fts FULL OUTER JOIN vec ON fts.id = vec.id
JOIN products p ON p.id = COALESCE(fts.id, vec.id)
ORDER BY rrf DESC, p.id LIMIT %(k)s'''

FTS_SQL = '''SELECT id FROM products, to_tsquery('simple', %(tsq)s) q
  WHERE fts_tokens @@ q AND (%(cat)s::text IS NULL OR category=%(cat)s)
  ORDER BY ts_rank(fts_tokens, q) DESC, id LIMIT %(k)s'''
VEC_SQL = '''SELECT id FROM products WHERE (%(cat)s::text IS NULL OR category=%(cat)s)
  ORDER BY embedding <=> %(vec)s::vector LIMIT %(k)s'''

def search(q, mode="hybrid", cat=None, k=5):
    prm = dict(tsq=tsq(q), vec=str(embed([q])[0]), cat=cat, k=k)
    sql = {"hybrid":HYBRID_SQL, "keyword":FTS_SQL, "vector":VEC_SQL}[mode]
    cur.execute(sql, prm); rows = cur.fetchall()
    ids = [r[0] for r in rows]
    return ids, rows

query = "laptop ASUS ROG RAM 16GB i7 bekas"
ids, rows = search(query, "hybrid", cat="Laptop", k=8)
res = pd.DataFrame(rows, columns=["id","name","price","rank_kw","rank_vec","rrf"]); res["rrf"] = res.rrf.astype(float).round(5)
print("Query:", query); print(res.to_string(index=False))


# ---------- Penjelasan: C.3 Evaluasi objektif: keyword-only vs vector-only vs hybrid -

[Gambar 9] C.2 Hybrid Search dengan RRF

Query: laptop ASUS ROG RAM 16GB i7 bekas
 id                                 name    price  rank_kw  rank_vec     rrf
 10     ASUS ROG Strix i7 RAM 16GB bekas  9100000        1         1 0.03279
  4     ASUS ROG Strix i5 RAM 16GB bekas  8450000        2         3 0.03200
 12     ASUS ROG Strix i7 RAM 32GB bekas 11700000        6         2 0.03128
  8      ASUS ROG Strix i7 RAM 8GB bekas  7800000        4         6 0.03078
  9      ASUS ROG Strix i7 RAM 16GB baru 14000000        5         5 0.03077
 34      ASUS VivoBook i7 RAM 16GB bekas  7150000        3         8 0.03058
 14 ASUS ROG Strix Ryzen 5 RAM 8GB bekas  7150000        7        15 0.02826
  2      ASUS ROG Strix i5 RAM 8GB bekas  7150000       12        11 0.02797


In [ ]:

shot("C3")
EVAL_Q = [
 ("laptop ASUS ROG RAM 16GB i7 bekas",       dict(brand="ASUS", line="ROG", ram_gb=16, cpu="i7", cond="bekas"), "Laptop"),
 ("notebook Lenovo Legion Ryzen 7 32GB baru", dict(brand="Lenovo", line="Legion", cpu="Ryzen 7", ram_gb=32, cond="baru"), "Laptop"),
 ("notebook Dell second RAM 8GB i5",          dict(brand="Dell", ram_gb=8, cpu="i5", cond="bekas"), "Laptop"),
 ("laptop HP Omen i7 RAM 32GB seken",         dict(brand="HP", line="Omen", cpu="i7", ram_gb=32, cond="bekas"), "Laptop"),
 ("sepatu lari wanita empuk Nike",            dict(brand="Nike", gender="Wanita"), "Sepatu"),
 ("hp Samsung RAM 8GB kamera 50MP",           dict(brand="Samsung", ram_gb=8), "Ponsel"),
]
PI = P.set_index("id")
def _eq(a, b):
    try: return float(a) == float(b)
    except (TypeError, ValueError): return str(a).lower() == str(b).lower()
def match_frac(pid, attrs):
    r, ok = PI.loc[pid], 0
    for k, v in attrs.items():
        x = r[k]
        ok += (str(v).lower() in str(x).lower()) if k == "line" else _eq(x, v)
    return ok / len(attrs)

rec = []
for q, attrs, cat in EVAL_Q:
    row = {"query": q}
    for mode in ["keyword","vector","hybrid"]:
        ids, _ = search(q, mode, cat=cat, k=5)
        fr = [match_frac(i, attrs) for i in ids] or [0]
        row[f"{mode}_match@5"] = round(float(np.mean(fr)), 3); row[f"{mode}_exact@1"] = int(bool(ids) and fr[0] == 1.0)
    rec.append(row)
EV = pd.DataFrame(rec); print(EV.to_string(index=False))
summary_hybrid = EV[[c for c in EV.columns if c!="query"]].mean().round(3)
print("\nRATA-RATA:"); print(summary_hybrid.to_string())


# ---------- Penjelasan: C.4 Verifikasi RRF & bukti indeks dipakai & tuning ----------

[Gambar 10] C.3 Evaluasi keyword, vector, hybrid

                                   query  keyword_match@5  keyword_exact@1  vector_match@5  vector_exact@1  hybrid_match@5  hybrid_exact@1
       laptop ASUS ROG RAM 16GB i7 bekas             0.84                1            0.80               1            0.84               1
notebook Lenovo Legion Ryzen 7 32GB baru             0.84                1            0.80               1            0.84               1
         notebook Dell second RAM 8GB i5             0.85                0            0.85               0            0.80               0
        laptop HP Omen i7 RAM 32GB seken             0.84                1            0.80               0            0.84               0
           sepatu lari wanita empuk Nike             0.70                1            0.70               1            0.70               1
          hp Samsung RAM 8GB kamera 50MP             0.70                1            0.60               1          

In [ ]:

shot("C4")
# (1) Verifikasi: skor RRF dari SQL = hitung ulang manual di Python
q = "laptop ASUS ROG RAM 16GB i7 bekas"
kw, _ = search(q, "keyword", cat="Laptop", k=20); vc, _ = search(q, "vector", cat="Laptop", k=20)
manual = {}
for lst in (kw, vc):
    for r, i in enumerate(lst, 1): manual[i] = manual.get(i, 0) + 1.0/(60+r)
top_manual = sorted(manual.values(), reverse=True)[:5]
_, rows = search(q, "hybrid", cat="Laptop", k=5); top_sql = [float(r[5]) for r in rows]
assert np.allclose(top_manual, top_sql), (top_manual, top_sql)
print("Verifikasi RRF SQL == hitung manual Python: LULUS ->", [round(x,5) for x in top_sql])

# (2) Bukti indeks dipakai (seqscan dimatikan agar planner memilih indeks pada tabel kecil)
cur.execute("SET enable_seqscan = off"); cur.execute("SET hnsw.ef_search = 100")
cur.execute("EXPLAIN SELECT id FROM products ORDER BY embedding <=> %s::vector LIMIT 10", (str(embed([q])[0]),))
plan_v = [r[0] for r in cur.fetchall()]
cur.execute("EXPLAIN SELECT id FROM products WHERE fts_tokens @@ to_tsquery('simple', %s)", (tsq(q),))
plan_k = [r[0] for r in cur.fetchall()]
print("\nRencana vektor :", [l.strip() for l in plan_v if "Scan" in l])
print("Rencana keyword:", [l.strip() for l in plan_k if "Scan" in l])
cur.execute("RESET enable_seqscan")

# (3) Tuning (Materi 1): ef_search besar = recall tinggi tetapi lebih lambat
def timeit(ef, n=30):
    cur.execute(f"SET hnsw.ef_search = {ef}"); vec = str(embed([q])[0]); t = time.time()
    for _ in range(n): cur.execute("SELECT id FROM products ORDER BY embedding <=> %s::vector LIMIT 10", (vec,)); cur.fetchall()
    return (time.time()-t)/n*1000
print({f"ef_search={e}": f"{timeit(e):.2f} ms/kueri" for e in (32, 100, 200)})
try:   # pgvector >= 0.8.0
    cur.execute("SET hnsw.iterative_scan = strict_order"); print("iterative_scan aktif (pgvector >= 0.8) -> aman untuk filter selektif")
except Exception as e:
    print("iterative_scan tidak tersedia pada versi pgvector ini (butuh >= 0.8.0)")


# ---------- Penjelasan: **Catatan desain untuk skala jutaan baris (konsep Materi 1, bukan dijalankan di Colab):** ----------


# ---------- Penjelasan: C.5 Pembobotan Alpha (Materi 1, bagian 5) sebagai pembanding RRF ----------


[Gambar 11] C.4 Verifikasi RRF, EXPLAIN, ef_search

Verifikasi RRF SQL == hitung manual Python: LULUS -> [0.03279, 0.032, 0.03128, 0.03078, 0.03077]

Rencana vektor : ['->  Index Scan using idx_products_hnsw on products  (cost=52.36..323.71 rows=268 width=16)']
Rencana keyword: ['Bitmap Heap Scan on products  (cost=36.41..106.73 rows=266 width=8)', '->  Bitmap Index Scan on idx_products_fts  (cost=0.00..36.34 rows=266 width=0)']
{'ef_search=32': '0.67 ms/kueri', 'ef_search=100': '0.81 ms/kueri', 'ef_search=200': '1.05 ms/kueri'}
iterative_scan tidak tersedia pada versi pgvector ini (butuh >= 0.8.0)


In [ ]:

shot("C5")
def norm(d):
    if not d: return {}
    lo, hi = min(d.values()), max(d.values())
    return {k: ((v - lo) / (hi - lo) if hi > lo else 1.0) for k, v in d.items()}

def alpha_search(q, alpha, cat=None, k=5):
    prm = dict(tsq=tsq(q), vec=str(embed([q])[0]), cat=cat, n=20)
    cur.execute("SELECT id, ts_rank(fts_tokens, qq) FROM products, to_tsquery('simple', %(tsq)s) qq "
                "WHERE fts_tokens @@ qq AND (%(cat)s::text IS NULL OR category=%(cat)s) "
                "ORDER BY 2 DESC, id LIMIT %(n)s", prm)
    kw = {i: float(s) for i, s in cur.fetchall()}
    cur.execute("SELECT id, 1 - (embedding <=> %(vec)s::vector) FROM products "
                "WHERE (%(cat)s::text IS NULL OR category=%(cat)s) "
                "ORDER BY embedding <=> %(vec)s::vector LIMIT %(n)s", prm)
    vc = {i: float(s) for i, s in cur.fetchall()}
    nk, nv = norm(kw), norm(vc)
    sc = {i: alpha * nv.get(i, 0) + (1 - alpha) * nk.get(i, 0) for i in set(nk) | set(nv)}
    return [i for i, _ in sorted(sc.items(), key=lambda x: (-x[1], x[0]))][:k]

rows = []
for a in [0.0, 0.25, 0.5, 0.75, 1.0]:
    fr, ex = [], []
    for q, attrs, cat in EVAL_Q:
        ids = alpha_search(q, a, cat); f = [match_frac(i, attrs) for i in ids] or [0]
        fr.append(np.mean(f)); ex.append(int(bool(ids) and f[0] == 1.0))
    rows.append({"metode": f"alpha = {a}", "match@5": round(float(np.mean(fr)), 3), "exact@1": round(float(np.mean(ex)), 3)})
rows.append({"metode": "RRF (k=60)", "match@5": float(summary_hybrid["hybrid_match@5"]), "exact@1": float(summary_hybrid["hybrid_exact@1"])})
alpha_summary = pd.DataFrame(rows); print(alpha_summary.to_string(index=False))
print("\nalpha=0 sama dengan keyword-only :", all(alpha_search(q, 0.0, c) == search(q, "keyword", cat=c, k=5)[0] for q, _, c in EVAL_Q))
print("(alpha=1 dapat berbeda urutan dari vector-only bila ada skor kembar/tie pada embedding)")


# ---------- Penjelasan: BAGIAN D — TUGAS 2: RAG Lengkap (ChromaDB + LangChain splitter + LLM opsional) ----------


[Gambar 12] C.5 Pembobotan alpha vs RRF

      metode  match@5  exact@1
 alpha = 0.0    0.795    0.833
alpha = 0.25    0.795    0.833
 alpha = 0.5    0.795    0.833
alpha = 0.75    0.795    0.667
 alpha = 1.0    0.758    0.667
  RRF (k=60)    0.787    0.667

alpha=0 sama dengan keyword-only : True
(alpha=1 dapat berbeda urutan dari vector-only bila ada skor kembar/tie pada embedding)


In [ ]:

DOCS = {
 "D1-Pengembalian": "Kebijakan Pengembalian Barang. Pembeli dapat mengajukan pengembalian barang (retur) paling lambat 7 hari setelah barang diterima. Barang harus belum dipakai dan kemasan lengkap beserta aksesori. Pengembalian dana diproses dalam 3 sampai 5 hari kerja setelah barang lolos pemeriksaan. Produk elektronik yang rusak akibat kelalaian pengguna tidak dapat diretur.",
 "D2-Garansi": "Ketentuan Garansi Produk. Laptop baru mendapat garansi resmi 12 bulan dan ponsel baru garansi resmi 12 bulan. Laptop bekas mendapat garansi toko selama 30 hari. Klaim garansi wajib menyertakan nomor seri (serial number) perangkat dan nota pembelian. Kerusakan akibat cairan atau benturan tidak termasuk garansi.",
 "D3-Pengiriman": "Layanan Pengiriman. Pengiriman reguler memakan waktu 2 sampai 4 hari kerja dan pengiriman ekspres 1 hari kerja. Ongkos kirim gratis untuk pembelian minimal Rp 250.000 ke wilayah Jawa. Nomor resi dikirim melalui aplikasi setelah barang diserahkan ke kurir. Barang yang hilang di perjalanan diganti penuh oleh toko.",
 "D4-Keamanan": "Keamanan Akun dan Deteksi Penipuan. Transaksi dengan nilai jauh di atas kebiasaan belanja pengguna akan dikenai verifikasi tambahan berupa OTP atau verifikasi wajah. Jika transaksi diblokir sementara, pengguna dapat menghubungi pusat bantuan. Tim peninjau manusia memeriksa setiap kasus dalam 1 kali 24 jam. Keputusan sistem otomatis tidak bersifat final dan dapat diajukan keberatan.",
 "D5-Pembayaran": "Metode Pembayaran. Tersedia transfer virtual account, dompet digital, dan kartu kredit. Cicilan kartu kredit 0 persen tersedia untuk tenor 3 sampai 12 bulan dengan minimal belanja Rp 1.000.000. Bayar di tempat (COD) hanya untuk nilai pesanan maksimal Rp 2.000.000.",
 "D6-Privasi": "Kebijakan Privasi Data. Data pribadi disimpan terenkripsi dan email serta nomor telepon disamarkan pada sistem analitik. Data transaksi disimpan selama 5 tahun sesuai ketentuan. Pengguna dapat meminta penghapusan data pribadi kapan saja. Atribut sensitif seperti agama dan suku tidak pernah dipakai untuk penilaian risiko.",
}
print({k: len(v) for k, v in DOCS.items()})


# ---------- Penjelasan: D.2 Chunking (LangChain) → Embedding → ChromaDB ----------



{'D1-Pengembalian': 359, 'D2-Garansi': 311, 'D3-Pengiriman': 313, 'D4-Keamanan': 384, 'D5-Pembayaran': 264, 'D6-Privasi': 322}


In [ ]:

shot("D2")
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=220, chunk_overlap=40, separators=["\n\n", ". ", " "])
chunks = []
for doc_id, text in DOCS.items():
    for j, ch in enumerate(splitter.split_text(text)):
        chunks.append({"cid": f"{doc_id}#{j}", "doc": doc_id, "text": ch.lstrip(". ").strip()})
print(f"{len(DOCS)} dokumen -> {len(chunks)} chunk | contoh:", chunks[1]["cid"], "|", chunks[1]["text"][:90], "...")

chroma = chromadb.Client()                                   # in-memory
try: chroma.delete_collection("kb_tokopintar")
except Exception: pass
col = chroma.create_collection("kb_tokopintar", metadata={"hnsw:space": "cosine"})
col.add(ids=[c["cid"] for c in chunks], documents=[c["text"] for c in chunks],
        embeddings=embed([c["text"] for c in chunks]), metadatas=[{"doc": c["doc"]} for c in chunks])
print("Chroma count:", col.count())


# ---------- Penjelasan: D.3 Retrieval: vector-only vs BM25-only vs hybrid (RRF) ----------


[Gambar 13] D.2 Chunking dan ChromaDB

6 dokumen -> 12 chunk | contoh: D1-Pengembalian#1 | Pengembalian dana diproses dalam 3 sampai 5 hari kerja setelah barang lolos pemeriksaan. P ...
Chroma count: 12


In [ ]:


shot("D3")
from rank_bm25 import BM25Okapi
def toks(t): return [w for w in re.findall(r"[a-z0-9]+", t.lower()) if w not in _STOP]
bm25 = BM25Okapi([toks(c["text"]) for c in chunks])
CID = [c["cid"] for c in chunks]; CDOC = {c["cid"]: c["doc"] for c in chunks}; CTXT = {c["cid"]: c["text"] for c in chunks}

def r_vector(q, n=10):
    res = col.query(query_embeddings=embed([q]), n_results=n); return res["ids"][0]
def r_bm25(q, n=10):
    sc = bm25.get_scores(toks(q)); order = np.argsort(-sc)[:n]; return [CID[i] for i in order if sc[i] > 0]
def rrf(lists, k=60):
    s = {}
    for lst in lists:
        for r, i in enumerate(lst, 1): s[i] = s.get(i, 0) + 1.0/(k + r)
    return [i for i, _ in sorted(s.items(), key=lambda x: -x[1])]
def r_hybrid(q, n=10): return rrf([r_vector(q, n), r_bm25(q, n)])

QA = [
 ("Berapa hari batas waktu retur barang?", "D1-Pengembalian"),
 ("Berapa lama garansi laptop bekas?", "D2-Garansi"),
 ("Kapan ongkir gratis berlaku?", "D3-Pengiriman"),
 ("Kenapa transaksi saya diblokir dan diminta verifikasi wajah?", "D4-Keamanan"),
 ("Apakah ada cicilan 0 persen kartu kredit?", "D5-Pembayaran"),
 ("Berapa lama data transaksi disimpan?", "D6-Privasi"),
 ("Maksimal nilai pesanan COD berapa?", "D5-Pembayaran"),
 ("Bagaimana klaim garansi dengan nomor seri?", "D2-Garansi"),
]
def evaluate(fn, k=3):
    hit, rr = [], []
    for q, gold in QA:
        docs = [CDOC[i] for i in fn(q)]
        hit.append(int(gold in docs[:k]))
        rr.append(next((1/(r+1) for r, d in enumerate(docs) if d == gold), 0))
    return round(np.mean(hit), 3), round(np.mean(rr), 3)
EVR = pd.DataFrame({m: evaluate(f) for m, f in [("vector", r_vector), ("bm25", r_bm25), ("hybrid_RRF", r_hybrid)]}, index=["Hit@3", "MRR"]).T
print(EVR)


# ---------- Penjelasan: D.4 Prompt Augmentation + Generation ----------


[Gambar 14] D.3 Evaluasi retrieval

            Hit@3    MRR
vector        1.0  1.000
bm25          1.0  0.938
hybrid_RRF    1.0  1.000


In [ ]:

shot("D4")
SYSTEM = ("Kamu adalah asisten layanan pelanggan TokoPintar. Jawab HANYA berdasarkan KONTEKS. "
          "Jika jawaban tidak ada di konteks, katakan 'Informasi tidak ditemukan di kebijakan kami.' Sebutkan sumber [dokumen].")
def build_prompt(q, ctx_ids):
    ctx = "\n".join(f"[{CDOC[i]}] {CTXT[i]}" for i in ctx_ids)
    return f"{SYSTEM}\n\nKONTEKS:\n{ctx}\n\nPERTANYAAN: {q}\nJAWABAN:"

def extractive(q, ctx_ids):
    qs = set(toks(q)); best, bs = None, -1
    for i in ctx_ids:
        for s in re.split(r"(?<=\.)\s+", CTXT[i]):
            sc = len(qs & set(toks(s)))
            if sc > bs: best, bs, src = s, sc, CDOC[i]
    return f"{best} [{src}]" if bs > 0 else "Informasi tidak ditemukan di kebijakan kami."

def answer(q, k=3):
    ids = r_hybrid(q)[:k]; prompt = build_prompt(q, ids)
    key = os.environ.get("OPENAI_API_KEY")
    if key:
        try:
            from openai import OpenAI
            r = OpenAI(api_key=key).chat.completions.create(model=os.environ.get("OPENAI_MODEL", "gpt-4o-mini"),
                    messages=[{"role": "user", "content": prompt}], temperature=0)
            return r.choices[0].message.content, ids, "OpenAI"
        except Exception as e:
            print("OpenAI gagal, pakai mode offline:", e)
    return extractive(q, ids), ids, "ekstraktif-offline"

for q in ["Berapa hari batas waktu retur barang?", "Apakah bisa bayar COD untuk pesanan Rp 3 juta?", "apa gunanya Kopdes bagi Indonesia?"]:
    a, ids, mode = answer(q); print(f"\nQ: {q}\nA ({mode}): {a}\nSumber chunk: {ids}")
print("\n--- Contoh prompt yang dikirim ke LLM ---\n" + build_prompt("Berapa hari batas waktu retur barang?", r_hybrid("Berapa hari batas waktu retur barang?")[:2]))


# ---------- Penjelasan: D.5 Anti-halusinasi ----------


# ---------- Penjelasan: D.6 Advanced RAG: Parent-Document Retrieval (Materi 1) ----------



[Gambar 15] D.4 Jawaban RAG


Q: Berapa hari batas waktu retur barang?
A (ekstraktif-offline): Pembeli dapat mengajukan pengembalian barang (retur) paling lambat 7 hari setelah barang diterima. [D1-Pengembalian]
Sumber chunk: ['D1-Pengembalian#0', 'D1-Pengembalian#1', 'D3-Pengiriman#0']

Q: Apakah bisa bayar COD untuk pesanan Rp 3 juta?
A (ekstraktif-offline): Bayar di tempat (COD) hanya untuk nilai pesanan maksimal Rp 2.000.000. [D5-Pembayaran]
Sumber chunk: ['D5-Pembayaran#1', 'D5-Pembayaran#0', 'D1-Pengembalian#1']

Q: apa gunanya Kopdes bagi Indonesia?
A (ekstraktif-offline): Informasi tidak ditemukan di kebijakan kami.
Sumber chunk: ['D3-Pengiriman#0', 'D4-Keamanan#0', 'D5-Pembayaran#0']

--- Contoh prompt yang dikirim ke LLM ---
Kamu adalah asisten layanan pelanggan TokoPintar. Jawab HANYA berdasarkan KONTEKS. Jika jawaban tidak ada di konteks, katakan 'Informasi tidak ditemukan di kebijakan kami.' Sebutkan sumber [dokumen].

KONTEKS:
[D1-Pengembalian] Kebijakan Pengembalian Bara

In [ ]:

shot("D6")
def retrieve_parent(q, k_docs=1):
    seen, out = set(), []
    for cid in r_hybrid(q):
        doc = CDOC[cid]
        if doc not in seen: seen.add(doc); out.append((cid, doc, DOCS[doc]))
        if len(out) == k_docs: break
    return out
q = "Berapa lama garansi laptop bekas?"
cid, doc, full = retrieve_parent(q)[0]
print("Pertanyaan    :", q)
print("Chunk terbaik :", cid, "->", CTXT[cid])
print("Dokumen induk :", doc, f"({len(full)} karakter, chunk hanya {len(CTXT[cid])} karakter)")
print(full)


# ---------- Penjelasan: BAGIAN E — Data Pipeline + Feature Store (Feast) untuk Deteksi Fraud ----------


[Gambar 16] D.6 Parent-Document Retrieval

Pertanyaan    : Berapa lama garansi laptop bekas?
Chunk terbaik : D2-Garansi#0 -> Ketentuan Garansi Produk. Laptop baru mendapat garansi resmi 12 bulan dan ponsel baru garansi resmi 12 bulan. Laptop bekas mendapat garansi toko selama 30 hari
Dokumen induk : D2-Garansi (311 karakter, chunk hanya 159 karakter)
Ketentuan Garansi Produk. Laptop baru mendapat garansi resmi 12 bulan dan ponsel baru garansi resmi 12 bulan. Laptop bekas mendapat garansi toko selama 30 hari. Klaim garansi wajib menyertakan nomor seri (serial number) perangkat dan nota pembelian. Kerusakan akibat cairan atau benturan tidak termasuk garansi.


In [ ]:

shot("E1")
def build_features(g):
    g = g.sort_values("created_at").copy(); s = g.set_index("created_at")["amount"]
    g["avg_amount_30d"] = s.rolling("30D", closed="left").mean().values
    g["n_tx_30d"]       = s.rolling("30D", closed="left", min_periods=0).count().values
    g["tx_count_1h"]    = s.rolling("1h",  closed="left", min_periods=0).count().values
    return g
feat = clean.groupby("user_id", group_keys=False).apply(build_features)
feat["event_timestamp"] = feat["created_at"]

# Snapshot "saat ini" per user (nilai terbaru untuk online serving) – definisi sama: semua riwayat SEBELUM waktu snapshot
SNAP = END + pd.Timedelta(minutes=1)
def snapshot(g):
    s = g.set_index("created_at")["amount"]; w30 = s[s.index > SNAP - pd.Timedelta(days=30)]; w1 = s[s.index > SNAP - pd.Timedelta(hours=1)]
    return pd.Series({"avg_amount_30d": w30.mean() if len(w30) else np.nan, "n_tx_30d": float(len(w30)), "tx_count_1h": float(len(w1))})
snap = clean.groupby("user_id").apply(snapshot).reset_index(); snap["event_timestamp"] = SNAP

cols = ["user_id", "event_timestamp", "avg_amount_30d", "n_tx_30d", "tx_count_1h"]
fs = pd.concat([feat[cols], snap[cols]], ignore_index=True).drop_duplicates(["user_id", "event_timestamp"], keep="last")
fs["created"] = SNAP
for c in ["avg_amount_30d", "n_tx_30d", "tx_count_1h"]: fs[c] = fs[c].astype("float32")
print(fs.shape); print(fs.head(4).to_string(index=False))


# ---------- Penjelasan: E.2 Definisikan & daftarkan Feature Store (Feast): Offline (Parquet) + Online (SQLite; di  ----------



[Gambar 17] E.1 Perhitungan fitur

(10850, 6)
 user_id           event_timestamp  avg_amount_30d  n_tx_30d  tx_count_1h                   created
       1 2026-08-12 17:39:50+00:00             NaN       0.0          0.0 2026-10-05 12:01:00+00:00
       1 2026-08-19 22:26:05+00:00   109895.000000       1.0          0.0 2026-10-05 12:01:00+00:00
       1 2026-08-21 08:03:43+00:00   126560.000000       2.0          0.0 2026-10-05 12:01:00+00:00
       1 2026-08-21 22:51:53+00:00   106191.335938       3.0          0.0 2026-10-05 12:01:00+00:00


In [ ]:

shot("E2")
from datetime import timedelta
from feast import FeatureStore, Entity, FeatureView, Field, FileSource
from feast.types import Float32
from feast.value_type import ValueType
import shutil, textwrap

REPO = "feast_repo"; shutil.rmtree(REPO, ignore_errors=True); os.makedirs(f"{REPO}/data")
open(f"{REPO}/feature_store.yaml", "w").write(textwrap.dedent('''
    project: tokopintar
    registry: data/registry.db
    provider: local
    online_store:
      type: sqlite
      path: data/online.db
    entity_key_serialization_version: 3
'''))
PARQ = os.path.abspath(f"{REPO}/data/user_stats.parquet"); fs.to_parquet(PARQ)

user = Entity(name="user_id", join_keys=["user_id"], value_type=ValueType.INT64)
src = FileSource(path=PARQ, timestamp_field="event_timestamp", created_timestamp_column="created")
user_stats_fv = FeatureView(name="user_stats", entities=[user], ttl=timedelta(days=90),
    schema=[Field(name="avg_amount_30d", dtype=Float32), Field(name="n_tx_30d", dtype=Float32), Field(name="tx_count_1h", dtype=Float32)],
    source=src)
store = FeatureStore(repo_path=REPO); store.apply([user, user_stats_fv])
print("Terdaftar:", [f.name for f in store.list_feature_views()])

# Materialize: pindahkan nilai TERBARU per user dari Offline -> Online store
store.materialize(start_date=(START - pd.Timedelta(days=1)).to_pydatetime(), end_date=(SNAP + pd.Timedelta(minutes=5)).to_pydatetime())
print(store.get_online_features(features=["user_stats:avg_amount_30d", "user_stats:n_tx_30d", "user_stats:tx_count_1h"], entity_rows=[{"user_id": 1}]).to_dict())


# ---------- Penjelasan: E.3 Training set dengan **Point-in-Time Join** (mencegah data leakage) ----------


[Gambar 18] E.2 Feast: apply dan materialize

Terdaftar: ['user_stats']
Materializing 1 feature views from 2026-08-05 12:00:00+00:00 to 2026-10-05 12:06:00+00:00 into the sqlite online store.

user_stats:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


{'user_id': [1], 'n_tx_30d': [15.0], 'avg_amount_30d': [91157.796875], 'tx_count_1h': [0.0]}


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:

shot("E3")
lab = clean[["user_id", "created_at", "amount", "is_fraud_label"]].rename(columns={"created_at": "event_timestamp"})
lab = lab.drop_duplicates(["user_id", "event_timestamp"], keep="last").reset_index(drop=True)
train_df = store.get_historical_features(entity_df=lab, features=["user_stats:avg_amount_30d", "user_stats:n_tx_30d", "user_stats:tx_count_1h"]).to_df()
print("Baris training set:", len(train_df))

# --- Bukti anti-leakage: bandingkan dengan join "naif" memakai nilai TERBARU (snapshot) ---
naive = lab.merge(snap[["user_id", "avg_amount_30d"]].rename(columns={"avg_amount_30d": "avg_naive"}), on="user_id")
cmp_ = train_df[["user_id", "event_timestamp", "avg_amount_30d"]].merge(naive[["user_id", "event_timestamp", "avg_naive"]], on=["user_id", "event_timestamp"])
diff = (cmp_.avg_amount_30d.fillna(-1) - cmp_.avg_naive.fillna(-1)).abs() > 1
print(f"Baris yang nilainya BERBEDA jika memakai join naif (= kebocoran informasi masa depan): {diff.mean()*100:.1f}%")


# ---------- Penjelasan: E.4 Latih model & evaluasi (split berbasis **waktu**, bukan acak) ----------



[Gambar 19] E.3 Point-in-Time Join



/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Baris training set: 10550
Baris yang nilainya BERBEDA jika memakai join naif (= kebocoran informasi masa depan): 100.0%


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:

shot("E4")
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score
td = train_df.copy()
med = td.avg_amount_30d.median()                                   # fallback imputasi (Materi 2: Fallback Mechanism)
td["avg_f"] = td.avg_amount_30d.fillna(med)
td["amount_ratio"] = td.amount / td.avg_f
td["freq_1h"] = td.tx_count_1h.fillna(0) + 1
td["S_rule"] = 0.05 * td.amount_ratio + 0.3 * td.freq_1h

cut = td.event_timestamp.quantile(0.7)
tr, te = td[td.event_timestamp <= cut], td[td.event_timestamp > cut]
print(f"Train: {len(tr)} (fraud {tr.is_fraud_label.sum()}) | Test: {len(te)} (fraud {te.is_fraud_label.sum()}) | cut {cut:%Y-%m-%d}")
X = ["amount_ratio", "freq_1h"]
lr = LogisticRegression(class_weight="balanced", max_iter=1000).fit(tr[X], tr.is_fraud_label)
te = te.assign(p=lr.predict_proba(te[X])[:, 1]); te["pred_lr"] = (te.p >= 0.5).astype(int); te["pred_rule"] = (te.S_rule >= 2.0).astype(int)
res = pd.DataFrame({
  "LogReg":       [roc_auc_score(te.is_fraud_label, te.p), precision_score(te.is_fraud_label, te.pred_lr), recall_score(te.is_fraud_label, te.pred_lr)],
  "Aturan S>=2":  [roc_auc_score(te.is_fraud_label, te.S_rule), precision_score(te.is_fraud_label, te.pred_rule), recall_score(te.is_fraud_label, te.pred_rule)],
}, index=["AUC", "Precision", "Recall"]).round(3); print(res)
print("Koefisien LogReg:", dict(zip(X, lr.coef_[0].round(3))))


# ---------- Penjelasan: E.5 Inference Service: fitur dari **Online Store** → skor → keputusan (+ pengukuran latens ----------


[Gambar 20] E.4 Evaluasi model

Train: 7385 (fraud 53) | Test: 3165 (fraud 69) | cut 2026-09-17
           LogReg  Aturan S>=2
AUC         1.000        1.000
Precision   0.958        1.000
Recall      1.000        0.319
Koefisien LogReg: {'amount_ratio': np.float64(2.328), 'freq_1h': np.float64(4.225)}


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:451: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  opt_res = optimize.minimize(
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:

shot("E5")
FEATS = ["user_stats:avg_amount_30d", "user_stats:n_tx_30d", "user_stats:tx_count_1h"]
def decide(user_id, amount, thr=2.0):
    f = store.get_online_features(features=FEATS, entity_rows=[{"user_id": int(user_id)}]).to_dict()
    def nz(x, d): return d if x is None or pd.isna(x) else x                              # fallback jika fitur NULL/NaN
    avg = nz(f["avg_amount_30d"][0], med)
    ratio = amount / avg; freq = nz(f["tx_count_1h"][0], 0) + 1
    S = 0.05 * ratio + 0.3 * freq
    return dict(user_id=user_id, amount=amount, avg_30d=round(avg), ratio=round(ratio, 1), freq_1h=int(freq), S=round(S, 2),
                keputusan="BLOKIR & MINTA VERIFIKASI WAJAH/OTP" if S >= thr else "IZINKAN")

uid = int(snap.dropna().user_id.iloc[0]); avg_u = snap.set_index("user_id").loc[uid, "avg_amount_30d"]
print("Normal  :", decide(uid, int(avg_u)))
print("Mencurigakan:", decide(uid, int(avg_u * 100)))

lat = []
for _ in range(200):
    t = time.perf_counter(); store.get_online_features(features=FEATS, entity_rows=[{"user_id": uid}]); lat.append((time.perf_counter()-t)*1000)
print(f"Latensi Online Store (SQLite lokal, terukur): median {np.median(lat):.2f} ms | p95 {np.percentile(lat, 95):.2f} ms")

# ---- Verifikasi rumus dengan contoh ShopeePay pada Materi 2 ----
avg = 3_000_000 / 60; ratio = 5_000_000 / avg; S = 0.05 * ratio + 0.3 * 2
assert abs(avg - 50_000) < 1e-9 and abs(ratio - 100) < 1e-9 and abs(S - 5.6) < 1e-9
print(f"\nVerifikasi contoh Materi 2: avg={avg:,.0f} | rasio={ratio:.0f} | S={S:.1f} >= 2.0 -> BLOKIR  (LULUS)")



[Gambar 21] E.5 Inference dan latensi

Normal  : {'user_id': 1, 'amount': 91157, 'avg_30d': 91158, 'ratio': 1.0, 'freq_1h': 1, 'S': 0.35, 'keputusan': 'IZINKAN'}
Mencurigakan: {'user_id': 1, 'amount': 9115780, 'avg_30d': 91158, 'ratio': 100.0, 'freq_1h': 1, 'S': 5.3, 'keputusan': 'BLOKIR & MINTA VERIFIKASI WAJAH/OTP'}
Latensi Online Store (SQLite lokal, terukur): median 0.23 ms | p95 0.34 ms

Verifikasi contoh Materi 2: avg=50,000 | rasio=100 | S=5.6 >= 2.0 -> BLOKIR  (LULUS)


In [ ]:

shot("A5")
# ===== A.5 Audit bias: False Positive Rate (pengguna SAH yang ditantang) per kelompok =====
aud = te.merge(users[["user_id", "city_tier"]], on="user_id"); legit = aud[aud.is_fraud_label == 0]
for nama, kol in [("Aturan S>=2", "pred_rule"), ("LogReg", "pred_lr")]:
    t = legit.groupby("city_tier")[kol].agg(["mean", "size"]).rename(columns={"mean": "FPR", "size": "n_sah"}); t["FPR"] = t.FPR.round(4)
    mx, mn = t.FPR.max(), t.FPR.min()
    print(f"\n[{nama}]"); print(t)
    if mx == 0: print("-> FPR = 0 di semua kelompok: tidak ada disparitas.")
    elif mn == 0: print(f"-> FPR tertinggi {mx:.4f}, terendah 0: rasio tak terhingga -> PERLU TINJAU (sampel kecil; cek manual & kalibrasi).")
    else:
        r = mx / mn; print(f"-> Rasio FPR tertinggi/terendah = {r:.2f} (ambang kebijakan internal 1.25) ->", "PERLU TINJAU" if r > 1.25 else "dalam batas")
print("\nCatatan: sampel kecil & data sintetis -> indikatif; di produksi audit berkala dengan data nyata.")


# ---------- Penjelasan: E.7 Monitoring Data Drift dengan Evidently AI (Materi 2) ----------


[Gambar 3] A.5 Audit bias (FPR per kelompok)



/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



[Aturan S>=2]
           FPR  n_sah
city_tier            
Tier1      0.0   1353
Tier2      0.0    994
Tier3      0.0    749
-> FPR = 0 di semua kelompok: tidak ada disparitas.

[LogReg]
              FPR  n_sah
city_tier               
Tier1      0.0015   1353
Tier2      0.0010    994
Tier3      0.0000    749
-> FPR tertinggi 0.0015, terendah 0: rasio tak terhingga -> PERLU TINJAU (sampel kecil; cek manual & kalibrasi).

Catatan: sampel kecil & data sintetis -> indikatif; di produksi audit berkala dengan data nyata.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:

shot("E7")
def drift_report(ref, cur_, name):
    ref, cur_ = ref[FEATS_DRIFT].reset_index(drop=True), cur_[FEATS_DRIFT].reset_index(drop=True)
    try:
        from evidently import Report
        from evidently.presets import DataDriftPreset
        snap = Report([DataDriftPreset()]).run(current_data=cur_, reference_data=ref)
        snap.save_html(f"{name}.html"); ms = snap.dict()["metrics"]; v = ms[0]["value"]
        cols = {m["config"]["column"]: (round(float(m["value"]), 3), float(m["config"]["threshold"]), bool(m["value"] >= m["config"]["threshold"]))
                for m in ms[1:] if m["config"].get("type", "").endswith("ValueDrift")}
        return int(v["count"]), float(v["share"]), "evidently API baru", cols
    except ImportError:
        from evidently.report import Report
        from evidently.metric_preset import DataDriftPreset
        r = Report(metrics=[DataDriftPreset()]); r.run(reference_data=ref, current_data=cur_)
        r.save_html(f"{name}.html"); res = r.as_dict()["metrics"][0]["result"]
        return int(res["number_of_drifted_columns"]), float(res["share_of_drifted_columns"]), "evidently API lama", {}

FEATS_DRIFT = ["amount_ratio", "freq_1h"]
drift_info = {}
try:
    n1, s1, ver, c1 = drift_report(tr, te, "drift_train_vs_test")
    sim = te.copy(); sim["amount_ratio"] = sim["amount_ratio"] * 3          # SIMULASI: nilai belanja naik 3x
    n2, s2, _, c2 = drift_report(tr, sim, "drift_simulasi")
    drift_info = {"versi": ver, "train_vs_test": (n1, round(s1, 2)), "simulasi": (n2, round(s2, 2))}
    print(f"[{ver}] train vs test      : {n1} dari {len(FEATS_DRIFT)} fitur drift (share {s1:.2f}) | per kolom (jarak, ambang, drift): {c1}")
    print(f"[{ver}] SIMULASI (rasio x3): {n2} dari {len(FEATS_DRIFT)} fitur drift (share {s2:.2f}) | per kolom (jarak, ambang, drift): {c2}")
    print("Aturan: dataset dianggap drift bila share >= 0,5 -> ALERT / pertimbangkan retraining")
    print("Laporan HTML: drift_train_vs_test.html dan drift_simulasi.html (buka dari panel Files Colab)")
except Exception as ex:
    print("Evidently tidak dapat dijalankan di lingkungan ini:", type(ex).__name__, ex)


# ---------- Penjelasan: E.8 Latihan Kasus Materi 2 (ShopeePay / USR_55): verifikasi hitungan ----------


[Gambar 22] E.7 Data drift (Evidently)



/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/evidently/legacy/tests/utils.py:183: DeprecationWarning: numpy.core is deprecated and has been renamed to numpy._core. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric.
  np.core.numeric.ScalarType = np.core.numeric.ScalarType + (ApproxValue, ApproxValueNoDict)  # type: 

[evidently API baru] train vs test      : 1 dari 2 fitur drift (share 0.50) | per kolom (jarak, ambang, drift): {'amount_ratio': (0.129, 0.1, True), 'freq_1h': (0.028, 0.1, False)}
[evidently API baru] SIMULASI (rasio x3): 1 dari 2 fitur drift (share 0.50) | per kolom (jarak, ambang, drift): {'amount_ratio': (1.068, 0.1, True), 'freq_1h': (0.028, 0.1, False)}
Aturan: dataset dianggap drift bila share >= 0,5 -> ALERT / pertimbangkan retraining
Laporan HTML: drift_train_vs_test.html dan drift_simulasi.html (buka dari panel Files Colab)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



In [ ]:


shot("E8")
print("Bagian 1.1: abaikan baris berstatus FAILED dan PENDING (belum final) sebelum dihitung model")
print("Bagian 1.2: A (rata-rata 30 hari) -> BATCH | B (gagal 1 jam) -> STREAMING | C (lokasi/perangkat 15 menit) -> STREAMING\n")
F1 = 3_000_000 / 30; F2 = 8_000_000 / F1; devices = 2
S = 0.02 * F2 + 1.5 * devices
assert F1 == 100_000 and F2 == 80 and abs(S - 4.6) < 1e-9
print(f"Bagian 2: F1 = {F1:,.0f} | F2 = {F2:.0f} | device = {devices} | S = 0,02 x {F2:.0f} + 1,5 x {devices} = {S:.1f}")
print("Keputusan:", "BLOKIR & MINTA VERIFIKASI WAJAH" if S >= 2.5 else "IZINKAN", "(ambang S >= 2,5)")



[Gambar 23] E.8 Latihan kasus Materi 2

Bagian 1.1: abaikan baris berstatus FAILED dan PENDING (belum final) sebelum dihitung model
Bagian 1.2: A (rata-rata 30 hari) -> BATCH | B (gagal 1 jam) -> STREAMING | C (lokasi/perangkat 15 menit) -> STREAMING

Bagian 2: F1 = 100,000 | F2 = 80 | device = 2 | S = 0,02 x 80 + 1,5 x 2 = 4.6
Keputusan: BLOKIR & MINTA VERIFIKASI WAJAH (ambang S >= 2,5)


In [ ]:

# ---------- Penjelasan: BAGIAN F — Ringkasan Hasil & Evaluasi (untuk Presentasi UTS) ----------



shot("F")
print("=== RINGKASAN ===")
print("Embedding        :", EMB_NAME)
print("\n[C] Hybrid Search PostgreSQL (rata-rata 6 query)"); print(summary_hybrid.to_string())
print("\n[D] RAG retrieval (8 pertanyaan)"); print(EVR.to_string())
print("\n[E] Fraud model (split waktu)"); print(res.to_string())
print("\n[C.5] Alpha weighting vs RRF"); print(alpha_summary.to_string(index=False))
print("\n[E.7] Data drift:", drift_info)
print("\nPrep DB:", report)
print("\nBatasan: data sintetis; embedding fallback bila model asli tidak tersedia; ts_rank != BM25 murni; latensi diukur pada SQLite lokal (bukan Redis produksi).")


[Gambar 24] F. Ringkasan hasil

=== RINGKASAN ===
Embedding        : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2

[C] Hybrid Search PostgreSQL (rata-rata 6 query)
keyword_match@5    0.795
keyword_exact@1    0.833
vector_match@5     0.758
vector_exact@1     0.667
hybrid_match@5     0.787
hybrid_exact@1     0.667

[D] RAG retrieval (8 pertanyaan)
            Hit@3    MRR
vector        1.0  1.000
bm25          1.0  0.938
hybrid_RRF    1.0  1.000

[E] Fraud model (split waktu)
           LogReg  Aturan S>=2
AUC         1.000        1.000
Precision   0.958        1.000
Recall      1.000        0.319

[C.5] Alpha weighting vs RRF
      metode  match@5  exact@1
 alpha = 0.0    0.795    0.833
alpha = 0.25    0.795    0.833
 alpha = 0.5    0.795    0.833
alpha = 0.75    0.795    0.667
 alpha = 1.0    0.758    0.667
  RRF (k=60)    0.787    0.667

[E.7] Data drift: {'versi': 'evidently API baru', 'train_vs_test': (1, 0.5), 'simulasi': (1, 0.5)}

Prep DB: {'baris_mentah': 11696, '

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

